In [2]:
import boto3
from dataplug import CloudObject
from dataplug.formats.geospatial.cog import CloudOptimizedGeoTiff, grid_partition_strategy
import time

session = boto3.Session()
creds = session.get_credentials().get_frozen_credentials()

s3_config = {
    "credentials": {
        "AccessKeyId": creds.access_key,
        "SecretAccessKey": creds.secret_key,
        "SessionToken": creds.token,
    },
    "region_name": session.region_name,
}


try:
    co = CloudObject.from_s3(CloudOptimizedGeoTiff, "s3://sentinel-cogs/sentinel-s2-l2a-cogs/10/S/FE/2024/9/S2A_10SFE_20240927_0_L2A/B04.tif", s3_config=s3_config)
    co.preprocess(force=True)  # You can skip this if needed
    chunk_options = [2, 4]

    for n_chunks in chunk_options:
        try:
            start = time.time()
            parts = co.partition(grid_partition_strategy, n_splits=n_chunks)
            end = time.time()
            duration = round(end - start, 2)
            print(f"  {n_chunks} chunks → {len(parts)} partitions in {duration} sec")
        except Exception as e:
            print(f"  Failed with {n_chunks} chunks: {e}")

except Exception as err:
    print(err)


  2 chunks → 4 partitions in 0.14 sec
  4 chunks → 16 partitions in 0.14 sec
